In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.disk import DiskImage, DiskBooleanMask
from mtrain.utils import mkdir
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.utils import show
from fastai.data.core import DataLoaders, default_device
from collections import defaultdict
from torchvision import tv_tensors
from torchvision.transforms import v2
from PIL import Image
from fastai.vision.all import (
    vision_learner,
    mobilenet_v3_small,
    mobilenet_v3_large,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
)

In [ ]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
ROCKS = NEG_MASKING_V1 / "rocks"
SAMPLES_MAPILLARY = NEG_MASKING_V1 / "samples_mapillary"
ROCKS_TXT_FILE = ROCKS / "clip_query_res.txt"
ROCKS_DATA_DIR = mkdir(ROCKS / "data")
ROCKS_CVAT_DIR = mkdir(ROCKS / "cvat")
ROCKS_DS_DIR = ROCKS / "classification" / "crop_level"

SAMPLES_MAPILLARY.exists(), ROCKS.exists()
BASE_MODEL_DIR = Path("../../datasets/models")

CACHE_CROP_LEVEL_PATH = ROCKS / "classification" / "cache_crop_level"

# function defs

In [ ]:
def _is_valid_dir(direc: Path):
    valid = (
        direc.is_dir()
        and (direc / "image.jpg").exists()
        and (direc / "mask.png").exists()
        and (direc / "meta.json").exists()
    )
    if valid:
        with open(direc / "meta.json") as f:
            content = json.load(f)
            if "crop_origin" not in content:
                return False
    return valid


def get_ds_dirs(ds_root, labels):
    res = []
    for label in labels:
        cls_root = ds_root / label
        print(cls_root.name)
        if not cls_root.is_dir():
            continue
        for d in cls_root.glob("*"):
            if _is_valid_dir(d):
                res.append(d)
    random.shuffle(res)
    return res


def _label_func(d: Path):
    return Path(d).parent.name


def _validate_labels_in_dirs(dirs, label_by_idx):
    for d in dirs:
        label = _label_func(d)
        if label not in label_by_idx:
            raise Exception(
                f"Label={label} not found for directory={d}. label_by_index={label_by_idx}"
            )


def get_area(d):
    return np.array(Image.open(d / "mask.png").convert("L")).sum()

# Model training

In [ ]:
LABELS = ["other", "trash"]
AREA_THRES = 5

In [ ]:
from mtrain.neg_mask.leveled_cropping import (
    load_crop_level_sample_from_directory,
    make_crop_level_pairs_v2,
)

CROP_LEVEL_PATH = ROCKS / "classification" / "crop_level"

DEST_DIR = mkdir(CROP_LEVEL_PATH.parent / "blurred")

for label in ["other", "trash"]:
    dirs = list((CROP_LEVEL_PATH / label).glob("*"))
    for p in tqdm(dirs):
        if (
            not p.is_dir()
            or not (p / "image.jpg").exists()
            or not (p / "source_dir" / "image.jpg").exists()
        ):
            continue
        try:
            sample = load_crop_level_sample_from_directory(p, 1024)
            level_pairs = make_crop_level_pairs_v2(sample, 130, 130 + 100, 1)
        except Exception as ex:
            print(f"WARN: failure in getting crop; dir={p.name} reason={ex}")

        crop, mask = level_pairs.pairs[0]
        dest_dir = mkdir(DEST_DIR / label / p.name)
        DiskImage.save(crop, dest_dir / "orig.jpg")
        DiskBooleanMask.save(mask, dest_dir / "mask.png")

In [ ]:
def get_padded_bbox_mask(mask, padding=10):
    # 1. Find the coordinates of all non-zero pixels
    coords = cv2.findNonZero(mask)
    if coords is None:
        return np.zeros_like(mask)

    # 2. Get the standard bounding box
    x, y, w, h = cv2.boundingRect(coords)
    img_h, img_w = mask.shape[:2]

    # 3. Apply padding with boundary constraints
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(img_w, x + w + padding)
    y2 = min(img_h, y + h + padding)

    # 4. Create the new mask
    padded_mask = np.zeros_like(mask)
    cv2.rectangle(padded_mask, (x1, y1), (x2, y2), 255, -1)

    return padded_mask

In [ ]:
def get_blurred_artifacts(
    img_path, mask_path, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
):
    img = cv2.imread(img_path)
    assert img is not None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = DiskBooleanMask.load(mask_path)
    new_mask = get_padded_bbox_mask(mask, bbox_pad)

    blurred = cv2.GaussianBlur(img, (blur_kernel_sz, blur_kernel_sz), blur_sigma)
    only_mask_unblurred = blurred.copy()

    new_mask = new_mask.astype(bool)
    only_mask_unblurred[new_mask] = img[new_mask]

    return {
        "mask": mask,
        "padded_mask": new_mask,
        "blurred": blurred,
        "unblurred": only_mask_unblurred,
        "img": img,
    }

In [ ]:
root_dir = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred"
)


def get_image_dirs(root_dir):
    dirs = []
    dirs = list((root_dir / "other").glob("*")) + list((root_dir / "trash").glob("*"))
    random.shuffle(dirs)
    return dirs

In [ ]:
dirs = get_image_dirs(root_dir)
idx = 0

In [ ]:
idx += 1
image_path = dirs[idx] / "orig.jpg"
mask_path = dirs[idx] / "mask.png"
b3_p5 = get_blurred_artifacts(
    image_path, mask_path, blur_kernel_sz=3, blur_sigma=1, bbox_pad=5
)
b5_p5 = get_blurred_artifacts(
    image_path, mask_path, blur_kernel_sz=5, blur_sigma=3, bbox_pad=5
)
b3_p10 = get_blurred_artifacts(
    image_path, mask_path, blur_kernel_sz=3, blur_sigma=1, bbox_pad=10
)
b5_p10 = get_blurred_artifacts(
    image_path, mask_path, blur_kernel_sz=5, blur_sigma=3, bbox_pad=10
)
show(
    [
        b3_p5["unblurred"],
        b5_p5["unblurred"],
    ]
)

# going with b3, p5

In [ ]:
import shutil

BLURRED_ROOT = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred"
)
IP_DS = BLURRED_ROOT / "clean"
OUT_DS = BLURRED_ROOT / "b3p5"


In [ ]:
def create_train_ds(ip_ds, out_ds):
    # remove the out ds first
    shutil.rmtree(out_ds, True)

    for label in ["other", "trash"]:
        label_d = ip_ds / label
        label_dirs = list(label_d.glob("*"))
        for d in tqdm(label_dirs):
            if not d.is_dir() or not (d / "orig.jpg").exists():
                continue

            arts = get_blurred_artifacts(
                d / "orig.jpg",
                d / "mask.png",
                blur_kernel_sz=3,
                blur_sigma=1,
                bbox_pad=5,
            )
            out_arr = arts["unblurred"]
            fname = f"{label}_{d.name}.jpg"
            out_dir = mkdir(out_ds / label)
            DiskImage.save(out_arr, out_dir / fname)


In [ ]:
# create_train_ds(IP_DS, OUT_DS)

In [ ]:
from fastai.vision.all import *

LOG_ROOT = (
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification"
)


def label_func(x):
    if x.startswith("other"):
        return "other"
    elif x.startswith("trash"):
        return "trash"
    else:
        raise Exception(f"bad file name {x}")


dls = ImageDataLoaders.from_name_func(
    LOG_ROOT,
    get_image_files(OUT_DS),
    valid_pct=0.2,
    seed=42,
    label_func=label_func,
    item_tfms=CropPad(130),
    batch_tfms=aug_transforms(),
    bs=8,
)

# learn = vision_learner(dls, resnet34, metrics=error_rate)
# learn.fine_tune(1)

In [ ]:
dls.show_batch()

In [ ]:
from torch import nn

learn = vision_learner(
    dls,
    resnet18,
    metrics=[accuracy, F1Score(average="macro")],
    loss_func=CrossEntropyLossFlat(weight=torch.tensor([1.0, 3.5])),
)

# 2. Modify the 'stem' (first layer) 
# Change kernel to 3x3 and stride to 1 to preserve detail
learn.model[0][0] = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# 3. Remove the initial MaxPool (set to Identity)
# Standard ResNet uses MaxPool immediately after the first conv; 
# for small images, this "blurs" your small area of focus.
learn.model[0][3] = nn.Identity()
learn = learn.remove_cb(ProgressCallback)

In [ ]:
learn = load_learner("/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification/unblurred-altered-resnet18-iter15.pkl")
learn.dls = dls

In [ ]:
learn.lr_find()

In [ ]:
learn.loss_func

In [ ]:
learn.fine_tune(1)

In [ ]:
learn.loss_func = FocalLossFlat(gamma=2)

In [ ]:
# learn.unfreeze()
learn.fit_one_cycle(5, lr_max=slice(1e-5,1e-4))

In [ ]:
learn.show_results()

In [ ]:
from fastai.interpret import ClassificationInterpretation
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
learn.export("unblurred-altered-resnet18-iter15")

In [ ]:
learn.fit_one_cycle(35)

In [ ]:
learn.export("unblurred-with_aug-iter_45.pkl")

## Dataset

## Convert to levelled crop sample dataset

In [ ]:
from mtrain.neg_mask.leveled_cropping import CropLevelPairs


def get_crops(d: Path):
    res = []
    for i in range(3):
        img = DiskImage.load(d / f"img_pair_{i}.jpg")
        mask = DiskBooleanMask.load(d / f"mask_pair_{i}.png")
        res.append((img, mask))
    return res


def save_crops(d, pair: CropLevelPairs):
    for i, (img, mask) in enumerate(pair.pairs):
        DiskImage.save(img, d / f"img_pair_{i}.jpg")
        DiskBooleanMask.save(mask, d / f"mask_pair_{i}.png")

In [ ]:
import json


def is_hard_dir(d: Path):
    if not (d / "meta.json").exists():
        return False
    with open(d / "meta.json") as f:
        meta = json.load(f)
        # if model disagree, it is hard, else not
        return meta.get("model_disagreement", False)


total_dirs = get_ds_dirs(ROCKS_DS_DIR, LABELS)
hard_dirs = [d for d in total_dirs if is_hard_dir(d)]
# twice as many hard_dirs
# total_dirs.extend(hard_dirs)
# total_dirs.extend(hard_dirs)

areas_and_dirs = [(get_area(d), d) for d in total_dirs]
dirs = [d for (a, d) in areas_and_dirs if a > AREA_THRES]

print(f"total directories scanned: {len(total_dirs)}")
# print(f"filtered directories: {len(dirs)}")


labels = [_label_func(d) for d in dirs]
train_dirs, valid_dirs = train_test_split(
    dirs, test_size=0.2, stratify=labels, random_state=42
)

In [ ]:
from mtrain.neg_mask.model.crop_level_dataset import (
    CropLevelDataset,
    CropLevelDataset2Chan,
)

In [ ]:
NUM_IN_CHANNELS = 8
train_ds = CropLevelDataset2Chan(
    train_dirs, LABELS, True, medium_center_prob=0.1, medium_pad=130
)
valid_ds = CropLevelDataset2Chan(
    valid_dirs, LABELS, False, medium_center_prob=1, medium_pad=130
)

In [ ]:
# shutil.rmtree(CACHE_CROP_LEVEL_PATH, True)
# persist_dataset_to_cache(train_ds, CACHE_CROP_LEVEL_PATH / "train")
# persist_dataset_to_cache(valid_ds, CACHE_CROP_LEVEL_PATH / "valid")

In [ ]:
# train_ds = DirectTensorLoadFromCacheDataset(CACHE_CROP_LEVEL_PATH / "train", NUM_IN_CHANNELS)
# valid_ds = DirectTensorLoadFromCacheDataset(CACHE_CROP_LEVEL_PATH / "train", NUM_IN_CHANNELS)

In [ ]:
dls = DataLoaders.from_dsets(
    train_ds,
    valid_ds,
    device=default_device(),
    num_workers=4,
    # pin_memory=True,
    persistent_workers=True,
)  # don't respawn workers each epoch)
LABELS

### Visualise single tensors

In [ ]:
from mtrain.utils import show, it_chain

img_and_masks = CropLevelDataset2Chan.denormalize(train_ds[14][0])
# print(train_ds[39][1])
show(it_chain(img_and_masks), (40, 40), ncols=4, axis="off")

In [ ]:
from mtrain.utils import show, it_chain

img_and_masks = CropLevelDataset2Chan.denormalize(train_ds[14][0])
# print(train_ds[39][1])
show(it_chain(img_and_masks), (40, 40), ncols=4, axis="off")

# Model train

- Current status: very imbalanced, need more points for trash and unknown

In [ ]:
counts = defaultdict(lambda: 0)
for d in dirs:
    counts[d.parent.name] += 1
counts

In [ ]:
from mtrain.neg_mask.model.learner import load_our_learner
from fastai.vision.all import resnet18, FocalLossFlat

counts = torch.tensor([6362, 2497], dtype=torch.float32)
weights = 1 / counts
weights = weights / weights.sum()  # normalize
weights = weights.to(default_device())
# loss_func = CrossEntropyLossFlat(weight=weights) if weights is not None else CrossEntropyLossFlat()
loss_func = FocalLossFlat(gamma=2)

learn = vision_learner(
    dls,
    resnet18,
    n_in=train_ds.num_channels,
    metrics=[accuracy, F1Score(average="macro")],
    loss_func=loss_func,
    n_out=len(LABELS),
    normalize=False,
)
learn = learn.remove_cb(ProgressCallback)

# learn = load_our_learner(dls, mobilenet_v3_large, None, LABELS)
# state_dict = torch.load('/Users/hariomnarang/Desktop/personal/roads/datasets/models/taco_pretrained_mask_classifier/mobilenet_v3_large_130x130_iter-20-pure-torch.pth')
# keys_to_remove = [k for k in state_dict.keys() if '1.8' in k]
# print("Removing:", keys_to_remove)
# for k in keys_to_remove:
#     del state_dict[k]

# learn.model.load_state_dict(state_dict, strict=False)

In [ ]:
model_dir = mkdir(DS / "models" / "trash_classification")
learn = learn.load((model_dir / "resnet18-size_130-chan_8-with_augs-iter_40").resolve())

In [ ]:
learn.fine_tune(5)

In [ ]:
learn.fit_one_cycle(10)

In [ ]:
model_dir = mkdir(DS / "models" / "trash_classification")
learn.save(
    (model_dir / "resnet18-size_130-chan_8-with_augs-iter_55-loss_focal").resolve()
)


In [ ]:
learn.fit_one_cycle(10)

In [ ]:
model_dir = mkdir(DS / "models" / "trash_classification")
learn.save(
    (model_dir / "resnet18-size_130-chan_8-with_augs-iter_65-loss_focal").resolve()
)


In [ ]:
learn.lr_find()

In [ ]:
model_path = (
    BASE_MODEL_DIR / "trash_classification" / "mobilenet_v3_large_chan-12_iter-12"
).resolve()
learn.save(model_path)

In [ ]:
for idx in range(len(train_ds)):
    train_ds[idx]

In [ ]:
model_dir = mkdir(DS / "models" / "trash_classification")
learn.save((model_dir / "mobilenet_v3_large_12_channel_v1").resolve())

# MobileNet Small Visualize

In [ ]:
model_path = (BASE_MODEL_DIR / "trash_classification" / "mobilenet_small").resolve()
learn = load_our_learner(dls, mobilenet_v3_small, weights, model_path)
all_preds, _, fp_idxs, fn_idxs = get_preds_for_valid_ds(learn, valid_ds)

In [ ]:
show_images(
    valid_ds, fp_idxs, all_preds, "False Positives (predicted trash, actually other)"
)
show_images(
    valid_ds, fn_idxs, all_preds, "False Negatives (predicted other, actually trash)"
)

# Mobilenet large visualize

In [ ]:
# model_path = (
#     BASE_MODEL_DIR / "trash_classification" / "mobilenet_v3_large_iter-20_v3"
# ).resolve()
# learn = load_our_learner(dls, mobilenet_v3_large, None, LABELS, model_path)
learner = learn.load(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification/mobilenet_v3_large_chan-12_iter-12"
)

## Classification interpretation

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_4 import LossWidget

In [ ]:
widget = LossWidget(learn, valid_ds, 100, dry_run=False, descending=False)

In [ ]:
widget.ui()

In [ ]:
from mtrain.neg_mask.model.show import get_preds_for_ds

all_valid_preds, all_valid_targs, _, all_valid_losses = get_preds_for_ds(
    learn,
    valid_ds,
)
# all_train_preds, all_train_targs, _, train_fp_idxs, train_fn_idxs, all_train_losses = get_preds_for_valid_ds(learn, train_ds)

In [ ]:
from mtrain.neg_mask.model.show import show_classification_report

show_classification_report(all_valid_preds, all_valid_targs, LABELS)

In [ ]:
from mtrain.neg_mask.model.show import show_classification_report

show_classification_report(all_valid_preds, all_valid_targs, LABELS)

In [ ]:
from mtrain.neg_mask.model.show import show_confusion_matrix_using_preds

show_confusion_matrix_using_preds(all_valid_preds, all_valid_targs, LABELS)

In [ ]:
# train_losses_and_idxes = list(reversed(sorted((loss,i) for i, loss in enumerate(all_train_losses))))
valid_losses_and_idxes = list(
    reversed(sorted((loss, i) for i, loss in enumerate(all_valid_losses)))
)

In [ ]:
import itertools

bad_train_dirs = ((idx, train_dirs[idx]) for _, idx in valid_losses_and_idxes)
img_and_masks = (
    (
        DiskImage.load(d / "image.jpg"),
        DiskBooleanMask.load(d / "mask.png"),
        (all_valid_preds[idx], all_valid_targs[idx]),
    )
    for (idx, d) in bad_train_dirs
)
img_and_masks = (r for r in img_and_masks if r[1].sum() > 0)
with_overlaid = (
    (img, mask, overlay_mask_on_img(img, mask.astype(bool)), pred)
    for (img, mask, pred) in img_and_masks
)

In [ ]:
print(next(with_overlaid)[-1])
show(next(with_overlaid)[:-1], (20, 20), ncols=3)

In [ ]:
pred_classes = all_valid_preds.argmax(dim=1)

In [ ]:
pred_classes[0], all_valid_preds[0]

In [ ]:
from mtrain.neg_mask.model.show import show_confusion_matrix_using_preds

show_confusion_matrix_using_preds(all_valid_preds, all_valid_targs, LABELS)

In [ ]:
predicted_vals = all_valid_preds.argmax(dim=1)

In [ ]:
len(predicted_vals), len(all_valid_targs)

In [ ]:
all_valid_preds[0]

In [ ]:
all_valid_preds.argmax(dim=1)[0] == 1

In [ ]:
from mtrain.neg_mask.model.show import show_images

LBL_OTHER, LBL_TRASH = 0, 1

pred_trash_is_other_idxs = [
    idx
    for idx, (predicted, actual, scores) in enumerate(
        zip(all_valid_preds.argmax(dim=1), all_valid_targs, all_valid_preds)
    )
    if predicted == LBL_TRASH and actual == LBL_OTHER
]
show_images(
    valid_ds,
    pred_trash_is_other_idxs,
    all_valid_preds,
    LABELS,
    "Predicted Trash but marked other",
)

In [ ]:
def mask_size_stats(pixel_counts):
    arr = np.array(pixel_counts)
    print(f"Count:    {len(arr)}")
    print(f"Mean:     {arr.mean():.1f}")
    print(f"Median:   {np.median(arr):.1f}")
    print(f"Std:      {arr.std():.1f}")
    print(f"Min:      {arr.min()}")
    print(f"Max:      {arr.max()}")
    print(f"25th pct: {np.percentile(arr, 25):.1f}")
    print(f"75th pct: {np.percentile(arr, 75):.1f}")
    print(f"95th pct: {np.percentile(arr, 95):.1f}")

    # fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # axes[0].hist(arr, bins=50, edgecolor='black')
    # axes[0].set_title('Distribution of Mask Pixel Counts')
    # axes[0].set_xlabel('Pixel Count')
    # axes[0].set_ylabel('Frequency')

    # axes[1].hist(arr, bins=50, edgecolor='black')
    # axes[1].set_yscale('log')
    # axes[1].set_title('Distribution (Log Scale)')
    # axes[1].set_xlabel('Pixel Count')
    # axes[1].set_ylabel('Frequency (log)')

    # for ax in axes:
    #     ax.axvline(np.median(arr), color='red', linestyle='--', label='Median')
    #     ax.axvline(arr.mean(), color='orange', linestyle='--', label='Mean')
    #     ax.legend()

    # plt.tight_layout()
    # plt.show()

In [ ]:
valid_fn_areas = [valid_ds[t][0][3].sum() for t in valid_fn_idxs]
valid_areas = [valid_ds[t][0][3].sum() for t in range(len(valid_ds))]
train_fn_areas = [train_ds[t][0][3].sum() for t in train_fn_idxs]
train_areas = [train_ds[t][0][3].sum() for t in range(len(train_ds))]

In [ ]:
mask_size_stats(train_areas)

In [ ]:
mask_size_stats(train_fn_areas)

In [ ]:
mask_size_stats(valid_areas)

In [ ]:
mask_size_stats(valid_fn_areas)

# Inference pipeline

In [ ]:
from mtrain.neg_mask.widget_2 import get_crops_for_image


def read_clip_file(path) -> list[tuple[str, Path]]:
    with open(path) as f:
        lines = f.readlines()
    imgs = [Path(line.split("\t")[1].strip()) for line in lines]
    dirs = [(path.stem, img.parent) for img in imgs]
    return dirs


paths = read_clip_file(NEG_MASKING_V1 / "trash" / "clip_delhi_litter.txt")

In [ ]:
idx = 31
plt.imshow(plt.imread(paths[idx][1] / "image.jpg"))

In [ ]:
d = paths[31][1]
img, mask = DiskImage.load(d / "image.jpg"), DiskBooleanMask.load(d / "m2.png")

In [ ]:
show([img, overlay_mask_on_img(img, mask.astype(bool))])

In [ ]:
def predict_trash(learn, img_mask_pairs, trash_pred_idx=0, threshold=0.25):
    from torch.utils.data import DataLoader as TorchDataLoader

    ds = MaskInferenceDataset(img_mask_pairs)
    dl = TorchDataLoader(ds, batch_size=64, shuffle=False, num_workers=0)

    learn.model.eval()
    learn.model.to(default_device())

    all_probs = []
    with torch.no_grad():
        for x in dl:
            x = x.to(default_device())
            probs = learn.model(x).softmax(dim=1)
            all_probs.append(probs.cpu())

    all_probs = torch.cat(all_probs)
    trash_probs = all_probs[:, trash_pred_idx]
    predicted_trash = trash_probs >= threshold

    return predicted_trash, trash_probs


def predict_and_reconstruct_mask(
    learn,
    image: np.ndarray,
    mask: np.ndarray,
    trash_pred_idx=0,
    bbox_pad=20,
    crop_pad=220,
):
    bboxes, imgs, masks = [], [], []
    for bbox, crop_img, crop_mask in get_crops_for_image(
        image, mask, bbox_pad, crop_pad
    ):
        bboxes.append(bbox)
        imgs.append(crop_img)
        masks.append(crop_mask)
    if not bboxes:
        return mask.copy().astype(np.float32)

    _, trash_probs = predict_trash(
        learn, list(zip(imgs, masks)), trash_pred_idx=trash_pred_idx, threshold=0
    )

    reconstructed = mask.copy().astype(np.float32)
    for bbox, prob in zip(bboxes, trash_probs):
        x, y, w, h = bbox.x, bbox.y, bbox.w, bbox.h
        region = reconstructed[y : y + h, x : x + w]
        region[region == 1] = prob.item()

    return reconstructed


def get_trash_mask(reconstructed: np.ndarray, threshold=0.25) -> np.ndarray:
    result = reconstructed.copy()
    result[(reconstructed > 0) & (reconstructed < threshold)] = 2
    result[reconstructed >= threshold] = 1
    return result

In [ ]:
new_mask = predict_and_reconstruct_mask(learn, img, mask)

In [ ]:
# generate the masks first
from fastai.vision.all import load_learner

learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)
learner50 = load_learner(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/iter_4_engulf_t009_more-skew-resnet18-50x50-v2/model.pkl"
)
SIZE = 100

In [ ]:
from mtrain.smallnet.unet.predict.strided import single

mask_100 = single.strided_predict_unet_only_mask(img, 100, learner100, [33, 66], 1)

In [ ]:
mask_50 = single.strided_predict_unet_only_mask(img, 50, learner50, [25], 1)

In [ ]:
O = overlay_mask_on_img
show(
    [
        O(img, mask.astype(bool)),
        O(img, mask_100.astype(bool)),
        O(img, mask_50.astype(bool)),
    ],
    (30, 30),
    ncols=3,
    axis="off",
)

In [ ]:
thres_25 = get_trash_mask(new_mask, 0.25)
thres_50 = get_trash_mask(new_mask, 0.5)
only_in_25 = (thres_25 == 1) & (thres_50 != 1)

show([thres_25, thres_50, only_in_25], (20, 20), 3, axis="off")

In [ ]:
show(
    [
        overlay_mask_on_img(img, mask.astype(bool)),
        mask,
        overlay_mask_on_img(img, get_trash_mask(new_mask) == 1),
        overlay_mask_on_img(img, get_trash_mask(new_mask, 0.5) == 1),
    ],
    (20, 20),
    axis="off",
)